# 高雄果菜市場蔬菜行情爬蟲（開發用 Notebook）

這份 notebook 用來邊看資料邊開發/除錯爬蟲邏輯，每一步都會印出中間結果方便檢查。

正式排程自動化執行的是 repo 內的 `for_crawler.py`，兩者核心邏輯完全一致；在這裡試出新邏輯後，記得同步更新 `for_crawler.py`。

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

URL = "https://www.khfv.com.tw/pagepub/AppContent.aspx?GP=GP04.01"
TABLE_ID = "WR1_1_WG1"

response = requests.get(URL, timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

table = soup.find('table', id=TABLE_ID)
if table is None:
    raise RuntimeError(f"找不到 ID 為 '{TABLE_ID}' 的表格，網站頁面可能已改版。")

print("成功抓到表格")

In [ ]:
all_rows = table.find_all('tr')
table_data = [
    [cell.text.strip() for cell in row.find_all(['td', 'th'])]
    for row in all_rows
]
table_data = [row for row in table_data if row]

column_headers = table_data[0]
data_rows = table_data[1:]

print(f"欄位：{column_headers}")
print(f"共抓到 {len(data_rows)} 筆交易紀錄")
data_rows[:5]  # 看前 5 筆原始資料長什麼樣子

In [ ]:
df = pd.DataFrame(data_rows, columns=column_headers)
df.head()

### 只留下需要的欄位，並把數字欄位轉成數值型別

In [ ]:
df_selected = df[['交易日期', '產品名稱', '均價', '交易量(公斤)']].copy()

df_selected['交易量(公斤)'] = pd.to_numeric(
    df_selected['交易量(公斤)'].astype(str).str.replace(',', '', regex=False),
    errors='coerce'
)
df_selected['均價'] = pd.to_numeric(df_selected['均價'], errors='coerce')

df_selected.head()

### 計算加權平均價

同一項產品在同一天可能有多筆交易紀錄（不同上中下價區間），依交易量加權平均後才是當日代表價格。

In [ ]:
df_weighted = (
    df_selected.groupby(['交易日期', '產品名稱'], as_index=False)
    .agg({
        '均價': lambda x: (x * df_selected.loc[x.index, '交易量(公斤)']).sum()
                          / df_selected.loc[x.index, '交易量(公斤)'].sum(),
        '交易量(公斤)': 'sum'
    })
    .rename(columns={'均價': '加權平均價(元/公斤)', '交易量(公斤)': '總交易量(公斤)'})
)
df_weighted['加權平均價(元/公斤)'] = df_weighted['加權平均價(元/公斤)'].round(6)

print(f"今日共 {len(df_weighted)} 項產品")
df_weighted.head(10)

### 抽查特定產品，確認加權平均算得合理

可以把 `冬瓜` 換成任何想確認的菜名。

In [ ]:
df_weighted[df_weighted['產品名稱'].str.contains('冬瓜', na=False)]

### 合併進歷史資料

先讀取現有的 `veg_prices_history.csv`，跟今天的資料合併，並依「交易日期＋產品名稱」去重（保留最新一筆）。

In [ ]:
df_history = pd.read_csv('veg_prices_history.csv')
print(f"合併前歷史資料共 {len(df_history)} 筆")

df_merged = pd.concat([df_history, df_weighted], ignore_index=True)
df_merged.drop_duplicates(subset=['交易日期', '產品名稱'], keep='last', inplace=True)

print(f"合併後共 {len(df_merged)} 筆")
df_merged.tail(10)

### 存檔

確認上面資料都沒問題後，才寫回 CSV。⚠️ 這一格會直接覆蓋 repo 裡的 CSV，測試階段建議先跳過或改存到別的檔名。

In [ ]:
df_weighted.to_csv('today_veg_prices.csv', index=False)
df_merged.to_csv('veg_prices_history.csv', float_format="%.6f", index=False)
print("已存檔")